In [ ]:
__import__('importlib').reload(__import__('nn'))
from nn import *
m = 256
def Cnv(i=m, h=m, r=15):
    return Sequential(
        Linear(i, h, 0),
        T(Conv1d(h, h, r, padding=r//2, groups=h, bias=0)),
    )
def At(i=m, h=m//16, H=m//16, g=16, G=1, r=16):
    return Sequential(
        Res(
            F(rope),
            Attn(i, h, H, g, G, r),
            Linear(G*g*(H or i), i, 0),
        ),
        LayerNorm(i, elementwise_affine=0),
    )
def Ff(i=m, h=m):
    return Sequential(
        Res(
            SwiGLU(i, h),
            Linear(h, i, 0),
        ),
        LayerNorm(i, elementwise_affine=0),
    )
def U(*_, i=m, h=m, s=2):
    return Res(
        Linear(i, h, 0),
        T(MaxPool1d(s)),
        *_,
        T(ConvTranspose1d(h, h, s, s, groups=h, bias=0)),
        Linear(h, i, 0),
    )
m = torch.compile(Sequential(
    Embedding(92, m),
    Cnv(),
    Ff(),
    U(
        Cnv(),
        Ff(),
        U(
            Cnv(),
            Ff(),
            U(
                Cnv(),
                Ff(),
                U(
                    Cnv(),
                    Ff(),
                    U(
                        Cnv(),
                        Ff(),
                        U(
                            Cnv(),
                            Ff()
                        ),
                        Cnv(),
                        Ff(),
                    ),
                    Cnv(),
                    Ff(),
                ),
                Cnv(),
                Ff(),
            ),
            Cnv(),
            Ff(),
        ),
        Cnv(),
        Ff(),
    ),
    Cnv(),
    Ff(),
    Linear(m, 92),
), fullgraph=True, dynamic=False).to('cuda',torch.bfloat16)
o = torch.optim.Adam(m.parameters(), 1/1024)
a = torch.from_numpy(np.memmap('py.92', mode='r'))
a = a.as_strided_((len(a)//4096-1,8192),(4096,1))
plot = Plot(2)
for y in range(1000000):
    t = a[torch.randint(len(a),[16])].to('cuda',torch.long)
    x = t.put(torch.randint(t.numel(),[t.numel()*3//4],device='cuda'),torch.randint(92,[t.numel()*3//4],device='cuda'))
    y = m(x)
    l = functional.cross_entropy(y.mT, t)
    (l*1024).backward()
    o.step()
    o.zero_grad()
    plot(l, lambda:(y.argmax(-1)==t).sum()/t.numel())